### What goes into Notebook 00:

* load every training CSV recursively from data/TrainingData
* assign label = parent_folder_name
* load and concatenate the 63 merged test CSVs
* verify the 39 feature columns align exactly
* normalize train/test labels into one canonical label space
* check class counts and imbalance
* check NaN/inf values
* build three target mappings:
  * 34-class: benign + individual attacks
  * 8-class: benign + 7 attack groups from the paper
  * binary: benign vs malicious

### Cell 1 -- Imports and Config

In [1]:
from pathlib import Path
import os
import re
import json
import math
import random

import numpy as np
import pandas as pd

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

TRAIN_ROOT = Path("../data/TrainingData")
TEST_ROOT = Path("../data/CIC_IOT_Dataset_2023")
ARTIFACT_DIR = Path("artifacts")
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_COLUMNS = [
    "Header_Length",
    "Protocol Type",
    "Time_To_Live",
    "Rate",
    "fin_flag_number",
    "syn_flag_number",
    "rst_flag_number",
    "psh_flag_number",
    "ack_flag_number",
    "ece_flag_number",
    "cwr_flag_number",
    "ack_count",
    "syn_count",
    "fin_count",
    "rst_count",
    "HTTP",
    "HTTPS",
    "DNS",
    "Telnet",
    "SMTP",
    "SSH",
    "IRC",
    "TCP",
    "UDP",
    "DHCP",
    "ARP",
    "ICMP",
    "IGMP",
    "IPv",
    "LLC",
    "Tot sum",
    "Min",
    "Max",
    "AVG",
    "Std",
    "Tot size",
    "IAT",
    "Number",
    "Variance",
]

TARGET_COLUMN_TEST = "Label"

print(f"Number of feature columns: {len(FEATURE_COLUMNS)}")

Number of feature columns: 39


### Cell 2 -- Collect File Lists

In [2]:
train_files = sorted(
    [
        p for p in TRAIN_ROOT.rglob("*.csv")
        if p.is_file()
    ]
)

test_files = sorted(
    [
        p for p in TEST_ROOT.glob("Merged*.csv")
        if p.is_file()
    ]
)

print(f"Training CSV files: {len(train_files)}")
print(f"Test CSV files: {len(test_files)}")

print("\nFirst 5 training files:")
for p in train_files[:5]:
    print(" -", p)

print("\nFirst 5 test files:")
for p in test_files[:5]:
    print(" -", p)

Training CSV files: 309
Test CSV files: 63

First 5 training files:
 - ../data/TrainingData/Backdoor_Malware/Backdoor_Malware.pcap.csv
 - ../data/TrainingData/Benign_Final/BenignTraffic.pcap.csv
 - ../data/TrainingData/Benign_Final/BenignTraffic1.pcap.csv
 - ../data/TrainingData/Benign_Final/BenignTraffic2.pcap.csv
 - ../data/TrainingData/Benign_Final/BenignTraffic3.pcap.csv

First 5 test files:
 - ../data/CIC_IOT_Dataset_2023/Merged01.csv
 - ../data/CIC_IOT_Dataset_2023/Merged02.csv
 - ../data/CIC_IOT_Dataset_2023/Merged03.csv
 - ../data/CIC_IOT_Dataset_2023/Merged04.csv
 - ../data/CIC_IOT_Dataset_2023/Merged05.csv


### Cell 3 -- Label Normalization Helpers 

In [3]:
def canonicalize_label(raw_label):
    if pd.isna(raw_label):
        return np.nan

    s = str(raw_label).strip()
    if s == "" or s.upper() == "NAN":
        return np.nan

    s = s.replace(".pcap.csv", "").replace(".csv", "").replace(".pcap", "")

    if re.fullmatch(r"BenignTraffic\d*", s, flags=re.IGNORECASE):
        return "BENIGN"
    if s.lower() == "benign_final":
        return "BENIGN"

    s = s.upper()
    s = s.replace("-", "_").replace(" ", "_")
    s = re.sub(r"__+", "_", s).strip("_")

    alias_map = {
        "BENIGNTRAFFIC": "BENIGN",
        "BENIGN_FINAL": "BENIGN",
        "BACKDOOR_MALWARE": "BACKDOOR_MALWARE",
        "BROWSERHIJACKING": "BROWSERHIJACKING",
        "COMMANDINJECTION": "COMMANDINJECTION",
        "SQLINJECTION": "SQLINJECTION",
        "UPLOADING_ATTACK": "UPLOADING_ATTACK",
        "XSS": "XSS",
        "MITM_ARPSPOOFING": "MITM_ARPSPOOFING",
        "DNS_SPOOFING": "DNS_SPOOFING",
        "DICTIONARYBRUTEFORCE": "DICTIONARYBRUTEFORCE",
        "RECON_HOSTDISCOVERY": "RECON_HOSTDISCOVERY",
        "RECON_OSSCAN": "RECON_OSSCAN",
        "RECON_PINGSWEEP": "RECON_PINGSWEEP",
        "RECON_PORTSCAN": "RECON_PORTSCAN",
        "VULNERABILITYSCAN": "VULNERABILITYSCAN",
        "MIRAI_GREETH_FLOOD": "MIRAI_GREETH_FLOOD",
        "MIRAI_GREIP_FLOOD": "MIRAI_GREIP_FLOOD",
        "MIRAI_UDPPLAIN": "MIRAI_UDPPLAIN",
    }

    return alias_map.get(s, s)


def map_label_to_group(label_34):
    if pd.isna(label_34):
        return np.nan
    if label_34 == "BENIGN":
        return "BENIGN"
    if label_34.startswith("DDOS_"):
        return "DDOS"
    if label_34.startswith("DOS_"):
        return "DOS"
    if label_34.startswith("RECON_") or label_34 == "VULNERABILITYSCAN":
        return "RECON"
    if label_34.startswith("MIRAI_"):
        return "MIRAI"
    if label_34 in {"DNS_SPOOFING", "MITM_ARPSPOOFING"}:
        return "SPOOFING"
    if label_34 == "DICTIONARYBRUTEFORCE":
        return "BRUTEFORCE"
    if label_34 in {
        "BACKDOOR_MALWARE",
        "BROWSERHIJACKING",
        "COMMANDINJECTION",
        "SQLINJECTION",
        "UPLOADING_ATTACK",
        "XSS",
    }:
        return "WEB"
    return "UNKNOWN"


def map_label_to_binary(label_34):
    if pd.isna(label_34):
        return np.nan
    return "BENIGN" if label_34 == "BENIGN" else "MALICIOUS"

### Cell 4 -- Schema Check Helper

In [4]:
def inspect_csv_schema(path: Path, expected_features, has_label: bool):
    header_df = pd.read_csv(path, nrows=0)
    cols = list(header_df.columns)

    expected = list(expected_features)
    if has_label:
        expected = expected + [TARGET_COLUMN_TEST]

    missing = [c for c in expected if c not in cols]
    extra = [c for c in cols if c not in expected]

    return {
        "path": str(path),
        "n_columns": len(cols),
        "columns": cols,
        "missing_columns": missing,
        "extra_columns": extra,
        "matches_expected": (len(missing) == 0 and len(extra) == 0),
    }

### Cell 5 -- Verify All Training Schemas

In [5]:
train_schema_rows = []
for path in train_files:
    train_schema_rows.append(inspect_csv_schema(path, FEATURE_COLUMNS, has_label=False))

train_schema_df = pd.DataFrame(train_schema_rows)

print("Training schema matches:")
print(train_schema_df["matches_expected"].value_counts(dropna=False))

bad_train_schema = train_schema_df.loc[~train_schema_df["matches_expected"]]
bad_train_schema.head(10)

Training schema matches:
matches_expected
True    309
Name: count, dtype: int64


,path,n_columns,columns,missing_columns,extra_columns,matches_expected


### Cell 6 -- Verify all test schemas

In [6]:
test_schema_rows = []
for path in test_files:
    test_schema_rows.append(inspect_csv_schema(path, FEATURE_COLUMNS, has_label=True))

test_schema_df = pd.DataFrame(test_schema_rows)

print("Test schema matches:")
print(test_schema_df["matches_expected"].value_counts(dropna=False))

bad_test_schema = test_schema_df.loc[~test_schema_df["matches_expected"]]
bad_test_schema.head(10)

Test schema matches:
matches_expected
True    63
Name: count, dtype: int64


,path,n_columns,columns,missing_columns,extra_columns,matches_expected


### Cell 7 -- Build File Manifests

In [7]:
def file_size_mb(path: Path) -> float:
    return path.stat().st_size / (1024 ** 2)


train_manifest = pd.DataFrame({
    "split": "train",
    "path": [str(p) for p in train_files],
    "file_name": [p.name for p in train_files],
    "folder_name": [p.parent.name for p in train_files],
    "label_raw": [p.parent.name for p in train_files],
    "label_34": [canonicalize_label(p.parent.name) for p in train_files],
    "label_8": [map_label_to_group(canonicalize_label(p.parent.name)) for p in train_files],
    "label_bin": [map_label_to_binary(canonicalize_label(p.parent.name)) for p in train_files],
    "file_size_mb": [file_size_mb(p) for p in train_files],
})

test_manifest = pd.DataFrame({
    "split": "test",
    "path": [str(p) for p in test_files],
    "file_name": [p.name for p in test_files],
    "file_size_mb": [file_size_mb(p) for p in test_files],
})

print(train_manifest.head())
print()
print(test_manifest.head())

   split                                               path  \
0  train  ../data/TrainingData/Backdoor_Malware/Backdoor...   
1  train  ../data/TrainingData/Benign_Final/BenignTraffi...   
2  train  ../data/TrainingData/Benign_Final/BenignTraffi...   
3  train  ../data/TrainingData/Benign_Final/BenignTraffi...   
4  train  ../data/TrainingData/Benign_Final/BenignTraffi...   

                   file_name       folder_name         label_raw  \
0  Backdoor_Malware.pcap.csv  Backdoor_Malware  Backdoor_Malware   
1     BenignTraffic.pcap.csv      Benign_Final      Benign_Final   
2    BenignTraffic1.pcap.csv      Benign_Final      Benign_Final   
3    BenignTraffic2.pcap.csv      Benign_Final      Benign_Final   
4    BenignTraffic3.pcap.csv      Benign_Final      Benign_Final   

           label_34 label_8  label_bin  file_size_mb  
0  BACKDOOR_MALWARE     WEB  MALICIOUS      0.633790  
1            BENIGN  BENIGN     BENIGN     71.621873  
2            BENIGN  BENIGN     BENIGN     58.4

### Cell 8 -- Inspect Training Label Space

In [8]:
print("Training 34-class labels:")
display(
    train_manifest.groupby("label_34")
    .agg(
        n_files=("path", "count"),
        total_size_mb=("file_size_mb", "sum")
    )
    .sort_values(["n_files", "total_size_mb"], ascending=False)
)

print("\nTraining 8-class labels:")
display(
    train_manifest.groupby("label_8")
    .agg(
        n_files=("path", "count"),
        total_size_mb=("file_size_mb", "sum")
    )
    .sort_values(["n_files", "total_size_mb"], ascending=False)
)

print("\nTraining binary labels:")
display(
    train_manifest.groupby("label_bin")
    .agg(
        n_files=("path", "count"),
        total_size_mb=("file_size_mb", "sum")
    )
    .sort_values(["n_files", "total_size_mb"], ascending=False)
)

Training 34-class labels:


,n_files,total_size_mb
label_34,,
MIRAI_GREETH_FLOOD,29,191.358942
DDOS_ICMP_FLOOD,27,1264.828220
MIRAI_UDPPLAIN,25,170.801708
MIRAI_GREIP_FLOOD,22,146.540701
DDOS_UDP_FLOOD,21,977.508106
DDOS_ICMP_FRAGMENTATION,20,95.096091
DDOS_TCP_FLOOD,18,804.744828
DOS_UDP_FLOOD,17,577.353579
DDOS_SYN_FLOOD,16,738.435287



Training 8-class labels:


,n_files,total_size_mb
label_8,,
DDOS,176,6120.439535
MIRAI,76,508.701351
DOS,38,1446.987138
WEB,6,4.887013
RECON,5,133.853135
BENIGN,4,216.878795
SPOOFING,3,95.118986
BRUTEFORCE,1,2.579045



Training binary labels:


,n_files,total_size_mb
label_bin,,
MALICIOUS,305,8312.566204
BENIGN,4,216.878795


### Cell 9 -- inspect test label space from actual rows 

In [9]:
def read_test_labels_only(path: Path) -> pd.DataFrame:
    df = pd.read_csv(path, usecols=[TARGET_COLUMN_TEST])
    df["label_raw"] = df[TARGET_COLUMN_TEST].astype(str)
    df["label_34"] = df["label_raw"].map(canonicalize_label)
    df["label_8"] = df["label_34"].map(map_label_to_group)
    df["label_bin"] = df["label_34"].map(map_label_to_binary)
    df["source_file"] = str(path)
    return df[["source_file", "label_raw", "label_34", "label_8", "label_bin"]]


test_label_parts = []
for path in test_files:
    part = read_test_labels_only(path)
    test_label_parts.append(part)

test_labels_df = pd.concat(test_label_parts, ignore_index=True)

print("Missing test labels:", test_labels_df["label_34"].isna().sum())
print("Unknown grouped labels:", test_labels_df["label_8"].eq("UNKNOWN").sum())
print("Unique 34-class labels:", sorted(test_labels_df["label_34"].dropna().unique()))

Missing test labels: 9
Unknown grouped labels: 0
Unique 34-class labels: ['BACKDOOR_MALWARE', 'BENIGN', 'BROWSERHIJACKING', 'COMMANDINJECTION', 'DDOS_ACK_FRAGMENTATION', 'DDOS_HTTP_FLOOD', 'DDOS_ICMP_FLOOD', 'DDOS_ICMP_FRAGMENTATION', 'DDOS_PSHACK_FLOOD', 'DDOS_RSTFINFLOOD', 'DDOS_SLOWLORIS', 'DDOS_SYNONYMOUSIP_FLOOD', 'DDOS_SYN_FLOOD', 'DDOS_TCP_FLOOD', 'DDOS_UDP_FLOOD', 'DDOS_UDP_FRAGMENTATION', 'DICTIONARYBRUTEFORCE', 'DNS_SPOOFING', 'DOS_HTTP_FLOOD', 'DOS_SYN_FLOOD', 'DOS_TCP_FLOOD', 'DOS_UDP_FLOOD', 'MIRAI_GREETH_FLOOD', 'MIRAI_GREIP_FLOOD', 'MIRAI_UDPPLAIN', 'MITM_ARPSPOOFING', 'RECON_HOSTDISCOVERY', 'RECON_OSSCAN', 'RECON_PINGSWEEP', 'RECON_PORTSCAN', 'SQLINJECTION', 'UPLOADING_ATTACK', 'VULNERABILITYSCAN', 'XSS']


### Cell 10 -- Compare Train vs Test Label Sets 

In [10]:
train_labels_34 = set(train_manifest["label_34"].dropna().unique())
test_labels_34 = set(test_labels_df["label_34"].dropna().unique())

train_only = sorted(train_labels_34 - test_labels_34)
test_only = sorted(test_labels_34 - train_labels_34)
shared = sorted(train_labels_34 & test_labels_34)

print(f"Shared 34-class labels: {len(shared)}")
print(f"Train-only labels: {len(train_only)}")
print(f"Test-only labels: {len(test_only)}")

print("\nTrain-only labels:")
print(train_only)

print("\nTest-only labels:")
print(test_only)

Shared 34-class labels: 34
Train-only labels: 0
Test-only labels: 0

Train-only labels:
[]

Test-only labels:
[]


### Cell 11 -- Save the Manifests

In [11]:
train_manifest_path = ARTIFACT_DIR / "train_manifest.csv"
test_manifest_path = ARTIFACT_DIR / "test_manifest.csv"
test_labels_path = ARTIFACT_DIR / "test_labels_full.csv"

train_manifest.to_csv(train_manifest_path, index=False)
test_manifest.to_csv(test_manifest_path, index=False)
test_labels_df.to_csv(test_labels_path, index=False)

print("Saved:")
print(" -", train_manifest_path)
print(" -", test_manifest_path)
print(" -", test_labels_path)

Saved:
 - artifacts/train_manifest.csv
 - artifacts/test_manifest.csv
 - artifacts/test_labels_full.csv


### Cell 12 -- Quick Dev Loaders for Small Experiments

In [12]:
def load_small_debug_sample(train_manifest: pd.DataFrame, rows_per_file: int = 2000) -> pd.DataFrame:
    parts = []

    for row in train_manifest.itertuples(index=False):
        path = Path(row.path)
        df = pd.read_csv(path, usecols=FEATURE_COLUMNS, nrows=rows_per_file)

        for col in FEATURE_COLUMNS:
            df[col] = pd.to_numeric(df[col], errors="coerce").astype("float32")

        df["label_34"] = row.label_34
        df["label_8"] = row.label_8
        df["label_bin"] = row.label_bin
        df["source_file"] = row.path
        parts.append(df)

    out = pd.concat(parts, ignore_index=True)
    return out


debug_df = load_small_debug_sample(train_manifest, rows_per_file=1000)
print(debug_df.shape)
debug_df.head()

(309000, 43)


,Header_Length,Protocol Type,Time_To_Live,Rate,fin_flag_number,syn_flag_number,rst_flag_number,psh_flag_number,ack_flag_number,ece_flag_number,...,AVG,Std,Tot size,IAT,Number,Variance,label_34,label_8,label_bin,source_file
0,13.200000,17.0,111.800003,21.654112,0.0,0.0,0.0,0.0,0.3,0.0,...,210.500000,415.549500,210.500000,0.046181,10.0,172681.390625,BACKDOOR_MALWARE,WEB,MALICIOUS,../data/TrainingData/Backdoor_Malware/Backdoor...
1,11.200000,17.0,63.500000,134.621811,0.0,0.1,0.0,0.0,0.0,0.0,...,473.600006,632.696228,473.600006,0.008186,10.0,400304.500000,BACKDOOR_MALWARE,WEB,MALICIOUS,../data/TrainingData/Backdoor_Malware/Backdoor...
2,13.600000,17.0,65.599998,211.662491,0.0,0.1,0.0,0.0,0.2,0.0,...,378.799988,508.763367,378.799988,0.004735,10.0,258840.171875,BACKDOOR_MALWARE,WEB,MALICIOUS,../data/TrainingData/Backdoor_Malware/Backdoor...
3,24.799999,6.0,84.400002,155.707336,0.0,0.0,0.0,0.4,0.7,0.0,...,291.700012,308.737946,291.700012,0.006422,10.0,95319.125000,BACKDOOR_MALWARE,WEB,MALICIOUS,../data/TrainingData/Backdoor_Malware/Backdoor...
4,10.400000,17.0,118.300003,105.440689,0.0,0.0,0.0,0.0,0.1,0.0,...,116.300003,75.052650,116.300003,0.009484,10.0,5632.899902,BACKDOOR_MALWARE,WEB,MALICIOUS,../data/TrainingData/Backdoor_Malware/Backdoor...


### Cell 13 -- chunk iterator for future notebooks

In [13]:
def iter_train_chunks(manifest_df: pd.DataFrame, chunksize: int = 250_000):
    for row in manifest_df.itertuples(index=False):
        path = Path(row.path)

        for chunk in pd.read_csv(path, usecols=FEATURE_COLUMNS, chunksize=chunksize):
            for col in FEATURE_COLUMNS:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float32")

            chunk["label_34"] = row.label_34
            chunk["label_8"] = row.label_8
            chunk["label_bin"] = row.label_bin
            chunk["source_file"] = row.path

            yield chunk


def iter_test_chunks(test_files, chunksize: int = 250_000):
    usecols = FEATURE_COLUMNS + [TARGET_COLUMN_TEST]

    for path in test_files:
        for chunk in pd.read_csv(path, usecols=usecols, chunksize=chunksize):
            for col in FEATURE_COLUMNS:
                chunk[col] = pd.to_numeric(chunk[col], errors="coerce").astype("float32")

            chunk["label_raw"] = chunk[TARGET_COLUMN_TEST].astype(str)
            chunk["label_34"] = chunk["label_raw"].map(canonicalize_label)
            chunk["label_8"] = chunk["label_34"].map(map_label_to_group)
            chunk["label_bin"] = chunk["label_34"].map(map_label_to_binary)
            chunk["source_file"] = str(path)

            yield chunk.drop(columns=[TARGET_COLUMN_TEST])